# Chess Play-Style Clustering
Unsupervised exploration of player archetypes using board features + clock features.

**Pipeline**
1. Feature engineering (same loader as `chess_elo_v_final`)
2. Per-player profile aggregation
3. Dimensionality reduction — t-SNE and UMAP
4. Clustering — K-Means and HDBSCAN
5. Archetype labelling & visualisation

> Toggle every behaviour from the **CONFIGURATION** cell — nowhere else.


In [ ]:
# pip install chess zstandard lightgbm umap-learn hdbscan pandas scikit-learn matplotlib seaborn joblib
import io, re, time, os, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import zstandard as zstd

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from joblib import Parallel, delayed

import umap
import hdbscan

from chess_features_final import extract_features_dataframe

warnings.filterwarnings('ignore')
print('All imports OK')


## ⚙️ CONFIGURATION — edit here only

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

DATA_PATH    = "../../data/villads_data/lichess_db_standard_rated_2017-11.pgn.zst"
MAX_GAMES    = 500_000      # reduce for faster iteration
RANDOM_STATE = 42
N_JOBS       = -1

# ── filters ───────────────────────────────────────────────────────────────────
FILTER_BASE_SECONDS  = 600   # None = all time controls
MIN_TOTAL_PLIES      = 12
EXCLUDE_TERMINATIONS = {"Time forfeit", "Abandoned", "Unterminated"}

# ── feature groups ────────────────────────────────────────────────────────────
FEATURE_GROUPS = {
    'structure':  True,
    'checks':     True,
    'captures':   True,
    'castling':   True,
    'style':      True,
    'clock':      True,
    'engine':     False,   # needs [%eval] tags in PGN
}

# ── per-player aggregation ────────────────────────────────────────────────────
# Each player appears in multiple games; we summarise their stats.
# Set False to use raw per-game rows (much larger matrix).
AGGREGATE_BY_PLAYER  = False   # True requires 'Username' column or similar
MIN_GAMES_PER_PLAYER = 5       # only used when AGGREGATE_BY_PLAYER=True

# ── dimensionality reduction ──────────────────────────────────────────────────
# t-SNE
TSNE_PERPLEXITY   = 40
TSNE_N_ITER       = 1000
TSNE_SAMPLE       = 30_000   # subsample for speed (None = all)

# UMAP
UMAP_N_NEIGHBORS  = 30
UMAP_MIN_DIST     = 0.05
UMAP_METRIC       = 'euclidean'

# ── clustering ────────────────────────────────────────────────────────────────
# K-Means
KMEANS_K          = 7        # number of clusters
RUN_ELBOW         = True     # plot elbow curve to help choose K
ELBOW_K_RANGE     = range(2, 14)

# HDBSCAN
HDBSCAN_MIN_CLUSTER_SIZE  = 500
HDBSCAN_MIN_SAMPLES       = 50
HDBSCAN_CLUSTER_SELECTION = 'eom'  # 'eom' or 'leaf'

# ── archetype labels ──────────────────────────────────────────────────────────
# Used to annotate cluster scatter plots after you've inspected the centroids.
# Keys are 0-indexed cluster IDs; leave empty and the plots use numeric IDs.
KMEANS_LABELS = {
    # 0: "Aggressive Attacker",
    # 1: "Blunder King",
    # 2: "Fast & Loose",
    # 3: "Positional Player",
    # 4: "Slow & Steady",
    # 5: "Endgame Specialist",
    # 6: "Solid Defender",
}

# ── output ────────────────────────────────────────────────────────────────────
VERBOSE_PLOTS  = True
SAVE_CSV       = True
OUTPUT_CSV     = f'chess_clusters_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'


## 1 · Load data

In [ ]:
games_list = []
dctx = zstd.ZstdDecompressor()

t0 = time.time()
with open(DATA_PATH, 'rb') as compressed_file:
    with dctx.stream_reader(compressed_file) as reader:
        text_stream = io.TextIOWrapper(reader, encoding='utf-8')
        current_game = {}
        for line in text_stream:
            line = line.strip()
            if line.startswith('['):
                tag = line.split(' ')[0][1:]
                val = line.split('"')[1]
                if tag in ('White', 'Black', 'WhiteElo', 'BlackElo',
                           'TimeControl', 'ECO', 'Termination', 'Result'):
                    current_game[tag] = val
            elif line.startswith('1.'):
                current_game['Moves'] = line
                if 'WhiteElo' in current_game and 'BlackElo' in current_game:
                    games_list.append(current_game)
                current_game = {}
                if len(games_list) >= MAX_GAMES:
                    break

df_raw = pd.DataFrame(games_list)
df_raw['WhiteElo'] = pd.to_numeric(df_raw['WhiteElo'], errors='coerce')
df_raw['BlackElo'] = pd.to_numeric(df_raw['BlackElo'], errors='coerce')
df_raw = df_raw.dropna(subset=['WhiteElo', 'BlackElo']).copy()
df_raw['WhiteElo'] = df_raw['WhiteElo'].astype(int)
df_raw['BlackElo'] = df_raw['BlackElo'].astype(int)
df_raw['Moves']    = df_raw['Moves'].fillna('').astype(str)

print(f'Loaded {len(df_raw):,} games in {time.time()-t0:.1f}s')

# ── optional time-control filter ──────────────────────────────────────────────
if FILTER_BASE_SECONDS is not None:
    _tc_base = (
        df_raw['TimeControl'].astype(str)
        .str.extract(r'^(\d+)\+')[0]
        .astype(float)
    )
    df_raw = df_raw[_tc_base == FILTER_BASE_SECONDS].copy().reset_index(drop=True)
    print(f'After TC filter ({FILTER_BASE_SECONDS}s): {len(df_raw):,} games')

if EXCLUDE_TERMINATIONS and 'Termination' in df_raw.columns:
    before = len(df_raw)
    df_raw = df_raw[~df_raw['Termination'].isin(EXCLUDE_TERMINATIONS)].copy().reset_index(drop=True)
    print(f'After termination filter: {len(df_raw):,} games (dropped {before-len(df_raw):,})')

# ── numeric time-control features ─────────────────────────────────────────────
if 'TimeControl' in df_raw.columns:
    _tc = df_raw['TimeControl'].astype(str).str.extract(r'^(\d+)\+(\d+)')
    df_raw['tc_base']      = pd.to_numeric(_tc[0], errors='coerce').fillna(0).astype(float)
    df_raw['tc_increment'] = pd.to_numeric(_tc[1], errors='coerce').fillna(0).astype(float)

df_raw.head(3)


## 2 · Clock features

In [ ]:
_CLK_RE = re.compile(r'\[%clk\s+(\d+):(\d+):(\d+)\]')

def parse_clock_features(moves_string: str, time_control: str = '?') -> dict:
    clocks = [int(h)*3600 + int(m)*60 + int(s)
              for h, m, s in _CLK_RE.findall(moves_string)]
    clk_w, clk_b = clocks[0::2], clocks[1::2]

    tc_m      = re.match(r'(\d+)\+(\d+)', str(time_control))
    base      = int(tc_m.group(1)) if tc_m else None
    increment = int(tc_m.group(2)) if tc_m else 0
    norm      = base if base else 1

    def _spent(seq, start):
        out, prev = [], start
        for c in seq:
            if prev is not None:
                s = prev - c + increment
                if s >= 0: out.append(s)
            prev = c
        return out

    sw = _spent(clk_w, base)
    sb = _spent(clk_b, base)

    def _stats(seq):
        if not seq: return 0., 0., 0.
        a = np.array(seq)
        return float(a.mean()), float(a.std()), float(a.max())

    def _trend(seq):
        if len(seq) < 2: return 0.0
        x = np.arange(len(seq))
        slope = np.polyfit(x, np.array(seq, dtype=float), 1)[0]
        return float(slope)

    def _window_mean(seq, n=10, which='first'):
        if not seq: return 0.0
        return float(np.mean(seq[:n])) if which == 'first' else float(np.mean(seq[-n:]))

    aw, sw2, mw = _stats(sw)
    ab, sb2, mb = _stats(sb)

    return {
        'avg_time_norm_white':         aw / norm,
        'std_time_norm_white':         sw2 / norm,
        'max_time_norm_white':         mw / norm,
        'time_pressure_white':         sum(1 for c in clk_w if c < 10),
        'opening_pace_norm_white':     (np.mean(sw[:10]) if sw else 0.) / norm,
        'clock_remaining_norm_white':  (clk_w[-1] / norm) if clk_w else 0.,
        'opening_time_norm_white':     _window_mean(sw, 10, 'first') / norm,
        'endgame_time_norm_white':     _window_mean(sw, 10, 'last') / norm,
        'time_spent_trend_norm_white': _trend(sw) / norm,
        'avg_time_norm_black':         ab / norm,
        'std_time_norm_black':         sb2 / norm,
        'max_time_norm_black':         mb / norm,
        'time_pressure_black':         sum(1 for c in clk_b if c < 10),
        'opening_pace_norm_black':     (np.mean(sb[:10]) if sb else 0.) / norm,
        'clock_remaining_norm_black':  (clk_b[-1] / norm) if clk_b else 0.,
        'opening_time_norm_black':     _window_mean(sb, 10, 'first') / norm,
        'endgame_time_norm_black':     _window_mean(sb, 10, 'last') / norm,
        'time_spent_trend_norm_black': _trend(sb) / norm,
    }

if FEATURE_GROUPS['clock']:
    tc_col = df_raw['TimeControl'] if 'TimeControl' in df_raw.columns else ['?']*len(df_raw)
    clock_records = Parallel(n_jobs=N_JOBS)(
        delayed(parse_clock_features)(m, t)
        for m, t in zip(df_raw['Moves'], tc_col)
    )
    df_clocks = pd.DataFrame(clock_records, index=df_raw.index)
    print('Clock features:', df_clocks.shape)
else:
    df_clocks = pd.DataFrame(index=df_raw.index)
    print('Clock features skipped')


## 3 · Board features

In [ ]:
t0       = time.time()
df_feats = extract_features_dataframe(df_raw, n_jobs=N_JOBS)
elapsed  = time.time() - t0
print(f'Extracted {len(df_feats):,} games in {elapsed:.1f}s  ({elapsed/len(df_feats)*1000:.1f} ms/game)')


## 4 · Build clustering feature matrix

In [ ]:
# ── feature lists ─────────────────────────────────────────────────────────────
_BOARD_FEATURES = {
    'structure': ['total_ply_count', 'material_balance_end', 'result_encoded'],
    'checks':    ['checks_given_white', 'checks_given_black',
                  'check_density_white', 'check_density_black'],
    'captures':  ['first_capture_move_white', 'first_capture_move_black',
                  'pawn_captures_total', 'piece_captures_total', 'capture_density'],
    'castling':  ['castle_move_white', 'castle_move_black'],
    'style':     ['consec_same_piece_white', 'consec_same_piece_black',
                  'queen_moves_before_10', 'white_territory_depth', 'black_territory_depth',
                  'promotions', 'en_passant_captures',
                  'legal_moves_white_move5', 'legal_moves_black_move5'],
    'engine':    ['acpl_white', 'inaccuracy_count_white', 'mistake_count_white',
                  'blunder_count_white', 'blunder_density_white',
                  'acpl_black', 'inaccuracy_count_black', 'mistake_count_black',
                  'blunder_count_black', 'blunder_density_black'],
}
_CLOCK_FEATURES = [
    'avg_time_norm_white', 'std_time_norm_white', 'max_time_norm_white',
    'time_pressure_white', 'opening_pace_norm_white', 'clock_remaining_norm_white',
    'opening_time_norm_white', 'endgame_time_norm_white', 'time_spent_trend_norm_white',
    'avg_time_norm_black', 'std_time_norm_black', 'max_time_norm_black',
    'time_pressure_black', 'opening_pace_norm_black', 'clock_remaining_norm_black',
    'opening_time_norm_black', 'endgame_time_norm_black', 'time_spent_trend_norm_black',
]

CLUSTER_FEATURES = []
for grp, cols in _BOARD_FEATURES.items():
    if FEATURE_GROUPS.get(grp, False):
        CLUSTER_FEATURES += cols
if FEATURE_GROUPS['clock']:
    CLUSTER_FEATURES += _CLOCK_FEATURES

# ── derived play-style composites ─────────────────────────────────────────────
# These are extra engineered features specifically useful for clustering
# player archetypes — they summarise cross-side patterns.

meta_cols = ['WhiteElo', 'BlackElo']
if 'tc_base' in df_raw.columns:
    meta_cols += ['tc_base', 'tc_increment']

df = (
    df_raw[meta_cols]
    .join(df_feats, how='inner')
    .join(df_clocks, how='inner')
)
df = df.replace([np.inf, -np.inf], np.nan)

# Symmetric (colour-blind) composites
df['avg_elo']             = (df['WhiteElo'] + df['BlackElo']) / 2
df['check_density_avg']   = df[['check_density_white','check_density_black']].mean(axis=1)
df['capture_agression']   = df['piece_captures_total'] / (df['total_ply_count'].clip(1))
df['pawn_aggression']     = df['pawn_captures_total']  / (df['total_ply_count'].clip(1))
df['consec_piece_avg']    = df[['consec_same_piece_white','consec_same_piece_black']].mean(axis=1)
df['time_pressure_avg']   = df[['time_pressure_white','time_pressure_black']].mean(axis=1)
df['avg_time_avg']        = df[['avg_time_norm_white','avg_time_norm_black']].mean(axis=1)
df['clock_remaining_avg'] = df[['clock_remaining_norm_white','clock_remaining_norm_black']].mean(axis=1)
df['opening_pace_avg']    = df[['opening_pace_norm_white','opening_pace_norm_black']].mean(axis=1)
df['time_variability']    = df[['std_time_norm_white','std_time_norm_black']].mean(axis=1)

COMPOSITE_FEATURES = [
    'avg_elo',
    'check_density_avg', 'capture_agression', 'pawn_aggression',
    'consec_piece_avg', 'queen_moves_before_10',
    'time_pressure_avg', 'avg_time_avg', 'clock_remaining_avg',
    'opening_pace_avg', 'time_variability',
    'total_ply_count', 'castle_move_white', 'castle_move_black',
    'white_territory_depth', 'black_territory_depth',
    'promotions', 'en_passant_captures',
    'material_balance_end',
]
if FEATURE_GROUPS['engine']:
    COMPOSITE_FEATURES += [
        'blunder_density_white', 'blunder_density_black',
        'acpl_white', 'acpl_black',
    ]

COMPOSITE_FEATURES = [c for c in COMPOSITE_FEATURES if c in df.columns]
print(f'Composite clustering features: {len(COMPOSITE_FEATURES)}')

# ── impute + scale ─────────────────────────────────────────────────────────────
X_raw = df[COMPOSITE_FEATURES].copy()
imputer = SimpleImputer(strategy='median')
X_imp   = imputer.fit_transform(X_raw)
scaler  = RobustScaler()
X_scaled = scaler.fit_transform(X_imp)

print(f'Clustering matrix: {X_scaled.shape}')


## 5 · K-Means — elbow + fit

In [ ]:
if RUN_ELBOW:
    inertias, sil_scores = [], []
    for k in ELBOW_K_RANGE:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
        labels = km.fit_predict(X_scaled)
        inertias.append(km.inertia_)
        # silhouette on a subsample to keep it fast
        _idx = np.random.default_rng(RANDOM_STATE).choice(len(X_scaled),
               size=min(5000, len(X_scaled)), replace=False)
        sil_scores.append(silhouette_score(X_scaled[_idx], labels[_idx]))
        print(f'  k={k}  inertia={km.inertia_:,.0f}  silhouette={sil_scores[-1]:.4f}')

    if VERBOSE_PLOTS:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        ax1.plot(list(ELBOW_K_RANGE), inertias, 'o-', color='#4e91d9')
        ax1.set_xlabel('k'); ax1.set_ylabel('Inertia')
        ax1.set_title('K-Means Elbow Curve'); ax1.grid(alpha=0.3)

        ax2.plot(list(ELBOW_K_RANGE), sil_scores, 'o-', color='#e05252')
        ax2.set_xlabel('k'); ax2.set_ylabel('Silhouette score')
        ax2.set_title('Silhouette Score vs k'); ax2.grid(alpha=0.3)

        plt.tight_layout(); plt.show()
        print(f'Best silhouette at k={list(ELBOW_K_RANGE)[np.argmax(sil_scores)]}')

# ── final K-Means fit ─────────────────────────────────────────────────────────
km_final = KMeans(n_clusters=KMEANS_K, random_state=RANDOM_STATE, n_init=20)
df['kmeans_cluster'] = km_final.fit_predict(X_scaled)
print(f'K-Means (k={KMEANS_K}) cluster sizes:')
print(df['kmeans_cluster'].value_counts().sort_index())


## 6 · HDBSCAN

In [ ]:
# HDBSCAN needs a lower-dimensional space — we use UMAP first (faster + better structure)
# so run UMAP before HDBSCAN.
print('Running UMAP for HDBSCAN pre-reduction...')
t0 = time.time()
reducer_hdb = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=0.0,          # 0 = tighter clusters, better for HDBSCAN
    n_components=10,       # 10-D embedding fed to HDBSCAN
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    low_memory=False,
)
X_umap_hdb = reducer_hdb.fit_transform(X_scaled)
print(f'UMAP (10-D) done in {time.time()-t0:.1f}s')

print('Running HDBSCAN...')
t0 = time.time()
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    cluster_selection_method=HDBSCAN_CLUSTER_SELECTION,
    metric='euclidean',
    core_dist_n_jobs=N_JOBS,
    prediction_data=True,
)
df['hdbscan_cluster'] = clusterer.fit_predict(X_umap_hdb)
n_found   = df['hdbscan_cluster'].nunique() - (1 if -1 in df['hdbscan_cluster'].values else 0)
n_noise   = (df['hdbscan_cluster'] == -1).sum()
print(f'HDBSCAN done in {time.time()-t0:.1f}s  |  clusters: {n_found}  noise: {n_noise:,}')
print(df['hdbscan_cluster'].value_counts().sort_index())


## 7 · t-SNE embedding

In [ ]:
print('Running t-SNE...')
t0 = time.time()

# Subsample if configured
if TSNE_SAMPLE and TSNE_SAMPLE < len(X_scaled):
    rng     = np.random.default_rng(RANDOM_STATE)
    ts_idx  = rng.choice(len(X_scaled), size=TSNE_SAMPLE, replace=False)
    X_tsne_input = X_scaled[ts_idx]
    df_tsne = df.iloc[ts_idx].copy().reset_index(drop=True)
    _subset = True
else:
    X_tsne_input = X_scaled
    df_tsne = df.copy().reset_index(drop=True)
    ts_idx  = np.arange(len(X_scaled))
    _subset = False

# Use PCA initialisation for reproducibility & speed
tsne = TSNE(
    n_components=2,
    perplexity=TSNE_PERPLEXITY,
    n_iter=TSNE_N_ITER,
    random_state=RANDOM_STATE,
    init='pca',
    learning_rate='auto',
    n_jobs=N_JOBS,
)
emb_tsne = tsne.fit_transform(X_tsne_input)
print(f't-SNE done in {time.time()-t0:.1f}s')


## 8 · UMAP embedding (2-D visualisation)

In [ ]:
print('Running UMAP (2-D)...')
t0 = time.time()

reducer_2d = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    n_components=2,
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    low_memory=False,
)
emb_umap = reducer_2d.fit_transform(X_scaled)
print(f'UMAP (2-D) done in {time.time()-t0:.1f}s')

df['umap_x'] = emb_umap[:, 0]
df['umap_y'] = emb_umap[:, 1]


## 9 · Visualise — t-SNE vs UMAP, coloured by cluster & Elo

In [ ]:
def _label(cluster_id: int, label_map: dict) -> str:
    return label_map.get(cluster_id, f'Cluster {cluster_id}')

# ── palette ───────────────────────────────────────────────────────────────────
PALETTE = [
    '#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
    '#42d4f4','#f032e6','#bfef45','#fabed4','#469990',
    '#dcbeff','#9a6324','#fffac8','#800000','#aaffc3',
]

def scatter_clusters(ax, x, y, labels, label_map, title, alpha=0.25, s=4):
    unique = sorted(set(labels))
    for i, c in enumerate(unique):
        mask  = labels == c
        color = '#aaaaaa' if c == -1 else PALETTE[i % len(PALETTE)]
        name  = 'Noise' if c == -1 else _label(c, label_map)
        ax.scatter(x[mask], y[mask], c=color, s=s, alpha=alpha,
                   rasterized=True, label=f'{name} (n={mask.sum():,})')
    ax.legend(loc='upper right', fontsize=7, markerscale=3)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

# ── t-SNE plots ───────────────────────────────────────────────────────────────
if VERBOSE_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    # K-Means on t-SNE
    scatter_clusters(axes[0],
                     emb_tsne[:,0], emb_tsne[:,1],
                     df_tsne['kmeans_cluster'].values,
                     KMEANS_LABELS, f't-SNE — K-Means (k={KMEANS_K})')

    # HDBSCAN on t-SNE
    scatter_clusters(axes[1],
                     emb_tsne[:,0], emb_tsne[:,1],
                     df_tsne['hdbscan_cluster'].values,
                     {}, 't-SNE — HDBSCAN')

    # Elo on t-SNE
    sc = axes[2].scatter(emb_tsne[:,0], emb_tsne[:,1],
                         c=df_tsne['avg_elo'].values,
                         cmap='RdYlGn', s=4, alpha=0.2, rasterized=True)
    plt.colorbar(sc, ax=axes[2], label='Avg Elo')
    axes[2].set_title('t-SNE — Avg Elo'); axes[2].set_xticks([]); axes[2].set_yticks([])

    plt.suptitle('t-SNE Embeddings', fontsize=14, y=1.01)
    plt.tight_layout(); plt.show()

    # ── UMAP plots ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))

    scatter_clusters(axes[0],
                     df['umap_x'].values, df['umap_y'].values,
                     df['kmeans_cluster'].values,
                     KMEANS_LABELS, f'UMAP — K-Means (k={KMEANS_K})')

    scatter_clusters(axes[1],
                     df['umap_x'].values, df['umap_y'].values,
                     df['hdbscan_cluster'].values,
                     {}, 'UMAP — HDBSCAN')

    sc = axes[2].scatter(df['umap_x'].values, df['umap_y'].values,
                         c=df['avg_elo'].values,
                         cmap='RdYlGn', s=4, alpha=0.2, rasterized=True)
    plt.colorbar(sc, ax=axes[2], label='Avg Elo')
    axes[2].set_title('UMAP — Avg Elo'); axes[2].set_xticks([]); axes[2].set_yticks([])

    plt.suptitle('UMAP Embeddings', fontsize=14, y=1.01)
    plt.tight_layout(); plt.show()


## 10 · Cluster profiling — centroids & radar charts

In [ ]:
# ── centroid table ────────────────────────────────────────────────────────────
_profile_cols = [
    'avg_elo', 'check_density_avg', 'capture_agression',
    'time_pressure_avg', 'avg_time_avg', 'clock_remaining_avg',
    'opening_pace_avg', 'time_variability',
    'queen_moves_before_10', 'consec_piece_avg',
    'total_ply_count', 'promotions', 'en_passant_captures',
]
_profile_cols = [c for c in _profile_cols if c in df.columns]

def cluster_profile(df_in: pd.DataFrame, cluster_col: str) -> pd.DataFrame:
    grp = df_in.groupby(cluster_col)[_profile_cols]
    return grp.mean().round(3)

km_profile  = cluster_profile(df, 'kmeans_cluster')
hdb_profile = cluster_profile(df[df['hdbscan_cluster'] >= 0], 'hdbscan_cluster')

print('=== K-Means cluster centroids ===')
display(km_profile)
print('\n=== HDBSCAN cluster centroids ===')
display(hdb_profile)


In [ ]:
# ── radar chart helper ────────────────────────────────────────────────────────
def radar_chart(profile_df: pd.DataFrame, title: str, label_map: dict = {}):
    # normalise each column to [0,1] for radar
    norm = (profile_df - profile_df.min()) / (profile_df.max() - profile_df.min() + 1e-9)
    cols  = list(norm.columns)
    N     = len(cols)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]   # close the polygon

    nrows = (len(norm) + 2) // 3
    fig   = plt.figure(figsize=(6 * min(3, len(norm)), 5 * nrows))

    for i, (cid, row) in enumerate(norm.iterrows()):
        ax = fig.add_subplot(nrows, min(3, len(norm)), i+1,
                             polar=True)
        vals = row.values.tolist() + [row.values[0]]
        color = '#aaaaaa' if cid == -1 else PALETTE[i % len(PALETTE)]
        ax.plot(angles, vals, color=color, linewidth=2)
        ax.fill(angles, vals, alpha=0.25, color=color)
        ax.set_xticks(angles[:-1])
        short = [c.replace('_norm','').replace('_avg','').replace('_white','')
                   .replace('total_ply_count','game_length')[:14]
                 for c in cols]
        ax.set_xticklabels(short, fontsize=7)
        name  = label_map.get(int(cid), f'Cluster {cid}') if cid != -1 else 'Noise'
        ax.set_title(name, fontsize=9, pad=12)
        ax.set_yticks([])

    plt.suptitle(title, fontsize=13)
    plt.tight_layout(); plt.show()

if VERBOSE_PLOTS:
    radar_chart(km_profile,  f'K-Means Play-Style Radar (k={KMEANS_K})', KMEANS_LABELS)
    radar_chart(hdb_profile, 'HDBSCAN Play-Style Radar')


## 11 · Archetype interpretation guide

After inspecting the centroids and radar charts, fill in `KMEANS_LABELS` in the CONFIG cell.

| Signature | Likely archetype |
|---|---|
| High `check_density_avg`, high `capture_aggression`, low `total_ply_count` | **Aggressive Attacker** |
| High `blunder_density` (engine on), high `time_pressure_avg` | **Blunder King / Time-Scrambler** |
| Low `avg_time_avg`, high `time_pressure_avg`, low `clock_remaining_avg` | **Fast & Loose** |
| Low `avg_time_avg`, low `time_pressure_avg`, high `avg_elo` | **Fast & Sharp** |
| High `avg_time_avg`, high `clock_remaining_avg`, long games | **Slow & Positional** |
| High `queen_moves_before_10`, low `castle_move_*` | **Gambit / Early Queen Rusher** |
| High `time_variability`, inconsistent `opening_pace` | **Erratic Thinker** |
| High `opening_pace_avg`, low `endgame_time_norm` | **Endgame Hoarder** |


## 12 · Save results

In [ ]:
if SAVE_CSV:
    # Attach t-SNE coords for the subsample; fill NaN for the rest
    df['tsne_x'] = np.nan
    df['tsne_y'] = np.nan
    df.loc[ts_idx if _subset else df.index, 'tsne_x'] = emb_tsne[:, 0]
    df.loc[ts_idx if _subset else df.index, 'tsne_y'] = emb_tsne[:, 1]

    save_cols = (
        ['WhiteElo', 'BlackElo', 'avg_elo']
        + _profile_cols
        + ['kmeans_cluster', 'hdbscan_cluster',
           'umap_x', 'umap_y', 'tsne_x', 'tsne_y']
    )
    save_cols = [c for c in save_cols if c in df.columns]
    df[save_cols].to_csv(OUTPUT_CSV, index=False)
    print(f'Saved → {OUTPUT_CSV}  ({len(df):,} rows)')
